# 1. Setup & Imports
This section imports required libraries, configures paths, and defines the experiment scope for FinSight LightGBM forecasting.

In [1]:
import sys
sys.path.append("../../../")

import json
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
from lightgbm import LGBMRegressor

from backend.data.data_loader import fetch_stock_data
from backend.data.preprocessor import FinSightPreprocessor

optuna.logging.set_verbosity(optuna.logging.INFO)

TICKER = "RELIANCE.NS"
START = "2020-01-01"
END = "2026-05-01"
TEST_START = "2025-01-01"

safe_ticker = ''.join(ch if ch.isalnum() else '_' for ch in TICKER)
ARTIFACTS_DIR = Path("./")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ticker: {TICKER}")
print(f"Date range: {START} to {END}")
print(f"Artifacts dir: {ARTIFACTS_DIR.resolve()}")

c:\Users\gaura\Desktop\FinSight\finsightvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ticker: RELIANCE.NS
Date range: 2020-01-01 to 2026-05-01
Artifacts dir: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm


# 2. Load Data
Load OHLCV data using FinSight's shared data loader and inspect shape and schema.

In [2]:
df = fetch_stock_data(TICKER, START, END)
print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (1567, 9)


Price,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread
Date,,,,,,,,,
2020-01-02,691.235535,704.470520,691.235535,701.887512,17710316,686.821289,0.017024,0.016881,13.234985
2020-01-03,700.835999,704.790527,696.264343,702.733276,20984698,687.648865,0.001205,0.001204,8.526184
2020-01-06,694.892883,698.504456,684.835205,686.435303,24519177,671.700684,-0.023192,-0.023465,13.669250
2020-01-07,694.435669,701.521790,691.921265,696.995850,16683622,682.034607,0.015385,0.015267,9.600525
2020-01-08,692.607056,701.498901,690.321228,691.761292,16047902,676.912354,-0.007510,-0.007539,11.177673


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1567 entries, 2020-01-02 to 2026-05-01
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1567 non-null   float64
 1   High          1567 non-null   float64
 2   Low           1567 non-null   float64
 3   Close         1567 non-null   float64
 4   Volume        1567 non-null   int64  
 5   Adj Close     1567 non-null   float64
 6   daily_return  1567 non-null   float64
 7   log_return    1567 non-null   float64
 8   hl_spread     1567 non-null   float64
dtypes: float64(8), int64(1)
memory usage: 122.4 KB


# 3. Feature Engineering
Apply FinSight preprocessing and optionally enrich the feature frame using graph features if the JSON artifact exists.
The target variable is next-day Close (`Close.shift(-1)`).

In [3]:
preprocessor = FinSightPreprocessor(ticker=TICKER)
processed_df = preprocessor.fit_transform(df)

graph_features_path = ARTIFACTS_DIR / "graph_features.json"
if graph_features_path.exists():
    with graph_features_path.open("r", encoding="utf-8") as f:
        graph_features = json.load(f)
    for k, v in graph_features.items():
        processed_df[k] = float(v)
    print(f"Loaded graph features from {graph_features_path}")
else:
    graph_features = {}
    print(f"No graph features found at {graph_features_path}; continuing without them.")

model_df = processed_df.copy()
model_df["target"] = model_df["Close"].shift(-1)
model_df = model_df.dropna(subset=["target"]).replace([np.inf, -np.inf], np.nan).dropna()

print("Final model frame shape:", model_df.shape)
display(model_df.head())
print("Feature columns (excluding target):", len([c for c in model_df.columns if c != "target"]))

No graph features found at graph_features.json; continuing without them.
Final model frame shape: (1547, 15)


,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread,Close_lag_1,Close_lag_5,Close_lag_10,rolling_mean_20,rolling_std_20,target
Date,,,,,,,,,,,,,,,
2020-01-29,-2.163245,-2.171218,-2.131566,-2.147750,0.447922,-2.132591,0.273794,0.282282,-0.783040,-2.159071,-2.022229,-2.020639,-2.036570,-1.426146,-2.218431
2020-01-30,-2.153547,-2.200000,-2.178682,-2.218431,0.291245,-2.200770,-1.410867,-1.421472,-0.432743,-2.143230,-2.034888,-1.993429,-2.045523,-1.217701,-2.281280
2020-01-31,-2.204485,-2.251787,-2.242940,-2.281280,1.116623,-2.261395,-1.289124,-1.296610,-0.194839,-2.213831,-2.045209,-1.909956,-2.057796,-0.929529,-2.332479
2020-02-03,-2.367291,-2.356144,-2.329434,-2.332479,0.846701,-2.310783,-1.080113,-1.082887,-0.537643,-2.276609,-2.074421,-2.004177,-2.069140,-0.600548,-2.252400
2020-02-04,-2.308320,-2.292414,-2.261356,-2.252400,0.490524,-2.233538,1.627040,1.614568,-0.620069,-2.327750,-2.142193,-2.001175,-2.078743,-0.488965,-2.209131


Feature columns (excluding target): 14


# 4. Train/Test Split
Create a chronological split with the last partition as test data (date-based split).

In [4]:
train_df = model_df[model_df.index < "2025-01-01"]
test_df = model_df[model_df.index >= "2025-01-01"]

feature_cols = [c for c in model_df.columns if c != "target"]
X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test = test_df[feature_cols], test_df["target"]

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Features:", len(feature_cols))

Train size: 1218
Test size: 329
Features: 14


# 5. Time Series Cross Validation
Run baseline cross-validation using `TimeSeriesSplit(n_splits=5)` and report fold RMSE.

In [5]:
tscv = TimeSeriesSplit(n_splits=5)
baseline_rmses = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
        device="gpu"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

    preds = model.predict(X_val)
    rmse = float(np.sqrt(mean_squared_error(y_val, preds)))
    baseline_rmses.append(rmse)
    print(f"Fold {fold} RMSE: {rmse:.6f}")

print(f"Mean CV RMSE: {np.mean(baseline_rmses):.6f}")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.00974857
Early stopping, best iteration is:
[93]	valid_0's l2: 0.00974393
Fold 1 RMSE: 0.098711
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.164455
[200]	valid_0's l2: 0.150721
Early stopping, best iteration is:
[193]	valid_0's l2: 0.150032
Fold 2 RMSE: 0.387340
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0048925
Early stopping, best iteration is:
[84]	valid_0's l2: 0.00482726
Fold 3 RMSE: 0.069479
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.116107
Early stopping, best iteration is:
[108]	valid_0's l2: 0.115168
Fold 4 RMSE: 0.339365
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0469504
Early stopping, best iteration is:
[96]	valid_0's l2: 0.0467519
Fold 5 RMSE: 0.216222
Mean CV RMSE: 0.222223


# 6. Hyperparameter Tuning (Optuna)
Tune LightGBM hyperparameters with 20 Optuna trials using time-series CV and GPU execution.

In [6]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "objective": "regression",
        "device": "gpu"
    }

    cv = TimeSeriesSplit(n_splits=3)
    rmses = []
    for tr_idx, val_idx in cv.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        pred = model.predict(X_val)
        rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
        rmses.append(rmse)

    print(f"Trial {trial.number} | RMSE: {float(np.mean(rmses)):.4f} | params: {trial.params}")
    return float(np.mean(rmses))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print("Best RMSE:", study.best_value)
print("Best params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

[I 2026-05-17 22:33:25,030] A new study created in memory with name: no-name-0722c567-ee1a-4bc9-a2bc-af39dcb0ce17


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[96]	valid_0's l2: 0.231112
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[46]	valid_0's l2: 0.0047802
Training until validation scores don't improve for 50 rounds


[I 2026-05-17 22:33:26,659] Trial 0 finished with value: 0.40498666085357077 and parameters: {'n_estimators': 788, 'learning_rate': 0.13081252798412732, 'max_depth': 3, 'subsample': 0.9678054992032926, 'colsample_bytree': 0.8102540900150063, 'min_child_weight': 10}. Best is trial 0 with value: 0.40498666085357077.


Early stopping, best iteration is:
[190]	valid_0's l2: 0.442332
Trial 0 | RMSE: 0.4050 | params: {'n_estimators': 788, 'learning_rate': 0.13081252798412732, 'max_depth': 3, 'subsample': 0.9678054992032926, 'colsample_bytree': 0.8102540900150063, 'min_child_weight': 10}
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[45]	valid_0's l2: 0.231479
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[17]	valid_0's l2: 0.00529452
Training until validation scores don't improve for 50 rounds


[I 2026-05-17 22:33:29,876] Trial 1 finished with value: 0.4101124544352557 and parameters: {'n_estimators': 550, 'learning_rate': 0.24453901972726413, 'max_depth': 9, 'subsample': 0.719023175879104, 'colsample_bytree': 0.8747467884892034, 'min_child_weight': 3}. Best is trial 0 with value: 0.40498666085357077.


...

Early stopping, best iteration is:
[74]	valid_0's l2: 0.451815
Trial 17 | RMSE: 0.4100 | params: {'n_estimators': 665, 'learning_rate': 0.2993963658580554, 'max_depth': 6, 'subsample': 0.892171515156214, 'colsample_bytree': 0.9434720159177314, 'min_child_weight': 2}
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[112]	valid_0's l2: 0.232302
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[28]	valid_0's l2: 0.00495224
Training until validation scores don't improve for 50 rounds


[I 2026-05-17 22:34:13,332] Trial 18 finished with value: 0.4047986982641452 and parameters: {'n_estimators': 901, 'learning_rate': 0.20534963226832478, 'max_depth': 4, 'subsample': 0.9277128650290356, 'colsample_bytree': 0.8401205754078725, 'min_child_weight': 5}. Best is trial 16 with value: 0.4016037293030659.


Early stopping, best iteration is:
[92]	valid_0's l2: 0.438305
Trial 18 | RMSE: 0.4048 | params: {'n_estimators': 901, 'learning_rate': 0.20534963226832478, 'max_depth': 4, 'subsample': 0.9277128650290356, 'colsample_bytree': 0.8401205754078725, 'min_child_weight': 5}
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[83]	valid_0's l2: 0.228227
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[35]	valid_0's l2: 0.00507779
Training until validation scores don't improve for 50 rounds


[I 2026-05-17 22:34:15,588] Trial 19 finished with value: 0.4071453668104326 and parameters: {'n_estimators': 110, 'learning_rate': 0.1656461676371161, 'max_depth': 5, 'subsample': 0.8924350646168671, 'colsample_bytree': 0.6021437264307581, 'min_child_weight': 2}. Best is trial 16 with value: 0.4016037293030659.


Did not meet early stopping. Best iteration is:
[94]	valid_0's l2: 0.452184
Trial 19 | RMSE: 0.4071 | params: {'n_estimators': 110, 'learning_rate': 0.1656461676371161, 'max_depth': 5, 'subsample': 0.8924350646168671, 'colsample_bytree': 0.6021437264307581, 'min_child_weight': 2}
Best RMSE: 0.4016037293030659
Best params:
  n_estimators: 871
  learning_rate: 0.2459402454215167
  max_depth: 4
  subsample: 0.8988349858268075
  colsample_bytree: 0.9181532462218961
  min_child_weight: 2


# 7. Final Model Training
Train the final model with best Optuna parameters on the full training set and evaluate on the holdout test set.

In [7]:
final_params = dict(best_params)
final_params.update({
    "random_state": 42,
    "objective": "regression",
    "device": "gpu"
})

final_model = LGBMRegressor(**final_params)
final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

test_pred = final_model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))
mae = float(mean_absolute_error(y_test, test_pred))
mape = float(np.mean(np.abs((y_test - test_pred) / y_test.replace(0, np.nan))) * 100)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.4f}%")

print("Retraining on full dataset...")
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])
final_model.fit(X_full, y_full, callbacks=[lgb.log_evaluation(100)])
print("Final model trained on full dataset")

# Auto-save
model_path = ARTIFACTS_DIR / f"{safe_ticker}_lightgbm_model.pkl"
best_params_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_best_params.json"
feature_cols_path = ARTIFACTS_DIR / f"{safe_ticker}_feature_columns.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print(f"AUTO SAVED as {safe_ticker}_*")
print(f"- Model: {model_path.resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[16]	valid_0's l2: 0.0084801
RMSE: 0.092087
MAE:  0.071412
MAPE: 14.9438%
Retraining on full dataset...
Final model trained on full dataset
AUTO SAVED as RELIANCE_NS_*
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\RELIANCE_NS_lightgbm_model.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\RELIANCE_NS_feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\RELIANCE_NS_lgbm_best_params.json


# 8. Feature Importance
Visualize the top 15 most important predictors from the trained LightGBM model.

In [8]:
importance = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

fig_imp = go.Figure(
    go.Bar(
        x=importance.values[::-1],
        y=importance.index[::-1],
        orientation="h",
        marker_color="#2563eb"
    )
)
fig_imp.update_layout(
    title=f"{TICKER} LightGBM Top 15 Feature Importance",
    template="plotly_white",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=600
)
out_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_feature_importance.html"
fig_imp.write_html(str(out_path))
print(f"Wrote feature importance to {out_path.resolve()}")

Wrote feature importance to C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\RELIANCE_NS_lgbm_feature_importance.html


# 9. Forecast Plot
Plot holdout-period actual vs predicted values and a 30-day future forecast using the latest feature row as a simple forward scenario.

In [9]:
future_steps = 30
last_row = X_test.iloc[[-1]] if len(X_test) > 0 else X_train.iloc[[-1]]
future_X = pd.concat([last_row] * future_steps, ignore_index=True)
future_pred = final_model.predict(future_X)

if isinstance(test_df.index, pd.DatetimeIndex):
    future_idx = pd.bdate_range(start=test_df.index[-1] + pd.tseries.offsets.BDay(1), periods=future_steps)
else:
    future_idx = pd.RangeIndex(start=len(test_df), stop=len(test_df) + future_steps)

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=test_df.index, y=y_test, mode="lines", name="Actual", line=dict(color="#111827", width=2)))
fig_fc.add_trace(go.Scatter(x=test_df.index, y=test_pred, mode="lines", name="Predicted", line=dict(color="#2563eb", width=2)))
fig_fc.add_trace(go.Scatter(x=future_idx, y=future_pred, mode="lines", name="30-Day Forecast", line=dict(color="#10b981", width=2, dash="dash")))
fig_fc.update_layout(
    title=f"{TICKER} Actual vs Predicted + 30-Day Forecast",
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="Close Price",
    height=600
)
out_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_forecast.html"
fig_fc.write_html(str(out_path))
print(f"Wrote forecast plot to {out_path.resolve()}")

Wrote forecast plot to C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\RELIANCE_NS_lgbm_forecast.html


# 10. Save Artifacts
Persist the trained model, preprocessing artifacts, feature columns, and best hyperparameters under the local artifacts directory.

In [10]:
model_path = ARTIFACTS_DIR / f"{safe_ticker}_lightgbm_model.pkl"
feature_cols_path = ARTIFACTS_DIR / f"{safe_ticker}_feature_columns.json"
best_params_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_best_params.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print("Saved artifacts:")
print(f"- Model: {model_path.resolve()}")
print(f"- Preprocessor scaler: {(ARTIFACTS_DIR / 'scaler.pkl').resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

Saved artifacts:
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\RELIANCE_NS_lightgbm_model.pkl
- Preprocessor scaler: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\scaler.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\RELIANCE_NS_feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\RELIANCE_NS_lgbm_best_params.json
